# Enhanced Image Generation with Gemini
This notebook generates enhanced, highly detailed versions of vehicle images from the `Vehicle Snapshots` directory.
It utilizes the new `google-genai` SDK and targets the Gemini 3 Flash model, adhering exclusively to a 5 RPM (Requests Per Minute) rate limit to avoid 429 status codes.

In [4]:
# Install the google-genai package and ipywidgets (to prevent tqdm warnings)
!pip install -q google-genai ipywidgets

In [ ]:
import os
import time

try:
    from google import genai
    from google.genai import types
except ImportError:
    print("Please run the cell above to install google-genai")
    raise

# Initialize the client with the provided API key
client = genai.Client(api_key='API_KEY')

# Utilizing the appropriate image-generation preview model for the Gemini 3 Flash family
model_id = 'gemini-3.1-flash-image-preview'

In [6]:
input_directory = os.path.join('Vehicle Snapshots', 'Bikes (Number-Plates On The Rear Mud-Guard)')
output_directory = os.path.join('Vehicle Snapshots', 'Enhanced Snapshots')
os.makedirs(output_directory, exist_ok=True)

print(f"Scanning Directory: {input_directory}...")

if not os.path.isdir(input_directory):
    print(f"Directory {input_directory} not found. Please check the path.")
else:
    for root, dirs, files in os.walk(input_directory):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
                file_path = os.path.join(root, file)
                new_filename = f"recreated_{file}"
                save_path = os.path.join(output_directory, new_filename)
                
                # Skip processing if the enhanced image already exists
                if os.path.exists(save_path):
                    print(f"Skipping {file_path}, enhanced image already exists.")
                    continue

                print(f"Loading {file_path}...")
                
                try:
                    # Upload the reference image to the AI platform
                    uploaded_img = client.files.upload(file=file_path)
                    
                    max_retries = 3
                    recreated_image_bytes = None
                    
                    for attempt in range(max_retries):
                        try:
                            # Generate content specifying IMAGE as the expected response modality
                            result = client.models.generate_content(
                                model=model_id,
                                contents=[
                                    uploaded_img, 
                                    "Recreate this overall vehicle, making the visual highly detailed and clear."
                                ],
                                config=types.GenerateContentConfig(
                                    response_modalities=["IMAGE"],
                                )
                            )
                            
                            # Extract the raw image byte data from the response
                            if result.parts:
                                for part in result.parts:
                                    if part.inline_data:
                                        recreated_image_bytes = part.inline_data.data
                                        break
                            
                            break # Success! Break out of the retry loop
                            
                        except Exception as e:
                            if ("503" in str(e) or "429" in str(e)) and attempt < max_retries - 1:
                                print(f"  -> Limit hit ({e}). Retrying attempt {attempt + 1} after 20 seconds...")
                                time.sleep(20)
                            else:
                                raise e 
                    
                    if recreated_image_bytes:
                        # Save the newly recreated image to the output directory
                        with open(save_path, "wb") as f:
                            f.write(recreated_image_bytes)
                            
                        print(f"Successfully recreated and saved image to: {save_path}")
                    else:
                        print(f"Failed to generate image data for {file}")
                    
                    # Delete file from Gemini storage to manage quotas
                    client.files.delete(name=uploaded_img.name)
                    
                    # RATE LIMIT BACKOFF FOR 5 RPM:
                    # 60 seconds / 5 requests = 12 seconds per request.
                    # A 12.5 second sleep guarantees staying safely under the 5 RPM strict boundary.
                    print("Waiting 12.5 seconds to respect the 5 RPM API limit...")
                    time.sleep(12.5)
                    
                except Exception as e:
                    print(f"Failed to process {file_path}: {e}")

print("Finished scanning and recreating files.")

Scanning Directory: Vehicle Snapshots\Bikes (Number-Plates On The Rear Mud-Guard)...
Loading Vehicle Snapshots\Bikes (Number-Plates On The Rear Mud-Guard)\T00217_bike.jpg...
  -> Limit hit (429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-3.1-flash-image\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-3.1-flash-image\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-3.1-flash-image\nPlease retry in 20.739782009s.', 'status': 'RESOURCE_EXHAUSTED', 'details': 

KeyboardInterrupt: 